In [1]:
# Code for 02_brfss_processing.ipynb

import pandas as pd
import numpy as np
import os
import pyreadstat

# --- Load and Merge the 3 yearly files ---
print("Processing BRFSS data...")
brfss_dfs = []
brfss_base_dir = '../data/01_raw/BRFSS'
for year in ['2020', '2021', '2022']:
    filename = f'LLCP{year}.XPT'
    path = os.path.join(brfss_base_dir, year, filename)
    print(f"  - Loading {filename}...")
    df, meta = pyreadstat.read_xport(path, encoding='latin1')
    brfss_dfs.append(df)

brfss_full_df = pd.concat(brfss_dfs, ignore_index=True)

# --- Select, Rename, and Harmonize ---
brfss_columns_to_keep = [
    'DIABETE4', '_AGEG5YR', 'SEXVAR', '_RACE', '_BMI5', 'GENHLTH', 
    '_SMOKER3', '_TOTINDA', 'CVDINFR4', 'CVDSTRK3',
]
brfss_selected_df = brfss_full_df[brfss_columns_to_keep].copy()

brfss_rename_dict = {
    'DIABETE4': 'Diabetes_Outcome', '_AGEG5YR': 'Age', 'SEXVAR': 'Gender', 
    '_RACE': 'Race_Ethnicity', '_BMI5': 'BMI', 'GENHLTH': 'General_Health', 
    '_SMOKER3': 'Smoking_Status', '_TOTINDA': 'Physical_Activity', 
    'CVDINFR4': 'History_Heart_Attack', 'CVDSTRK3': 'History_Stroke'
}
brfss_clean_df = brfss_selected_df.rename(columns=brfss_rename_dict)

# --- Harmonize the values ---
brfss_clean_df['BMI'] = brfss_clean_df['BMI'] / 100.0
brfss_clean_df['Diabetes_Outcome'] = brfss_clean_df['Diabetes_Outcome'].replace({1: 1, 3: 0, 4: 0, 2: 0, 7: np.nan, 9: np.nan})
age_map = { 1: 21.5, 2: 27.5, 3: 32.5, 4: 37.5, 5: 42.5, 6: 47.5, 7: 52.5, 8: 57.5, 9: 62.5, 10: 67.5, 11: 72.5, 12: 77.5, 13: 80.0, 14: np.nan }
brfss_clean_df['Age'] = brfss_clean_df['Age'].map(age_map)
brfss_clean_df['Gender'] = brfss_clean_df['Gender'].replace({1: 0, 2: 1})
brfss_clean_df['Smoking_Status'] = brfss_clean_df['Smoking_Status'].map({1: 1, 2: 1, 3: 1, 4: 0, 9: np.nan})
brfss_clean_df['General_Health'] = brfss_clean_df['General_Health'].replace({7: np.nan, 9: np.nan})
for col in ['Physical_Activity', 'History_Heart_Attack', 'History_Stroke']:
    brfss_clean_df[col] = brfss_clean_df[col].replace({1: 1, 2: 0, 7: np.nan, 9: np.nan})

# --- Save the processed file ---
output_path = '../data/03_processed/brfss_final.csv'
brfss_clean_df.to_csv(output_path, index=False)

print("\nBRFSS data has been fully harmonized and saved.")
brfss_clean_df.head()

Processing BRFSS data...
  - Loading LLCP2020.XPT...
  - Loading LLCP2021.XPT...
  - Loading LLCP2022.XPT...

BRFSS data has been fully harmonized and saved.


,Diabetes_Outcome,Age,Gender,Race_Ethnicity,BMI,General_Health,Smoking_Status,Physical_Activity,History_Heart_Attack,History_Stroke
0,1.0,57.5,1.0,1.0,16.60,2.0,1.0,1.0,0.0,0.0
1,0.0,67.5,1.0,2.0,29.18,3.0,NaN,1.0,0.0,0.0
2,0.0,67.5,1.0,2.0,NaN,3.0,0.0,1.0,0.0,0.0
3,0.0,80.0,1.0,1.0,NaN,1.0,0.0,0.0,0.0,0.0
4,0.0,80.0,1.0,1.0,20.34,2.0,0.0,1.0,0.0,1.0
